# SyncOps AI — Setup B: Google Colab Local Consensus Verifier Server

This notebook runs the **Gemma 4 Edge (E4B-it)** model locally in a Google Colab GPU environment (Tesla T4 or similar) using **Ollama**, and exposes the API endpoint to your backend application running elsewhere.

### Why Gemma 4 E4B?
- **Efficient Active-Inference**: Uses only 4.5B active parameters per token generation, giving fast output speed.
- **Tesla T4 Compatibility**: Quantized versions consume only ~6-10 GB VRAM, leaving safe headroom for context cache.
- **Native Agent Compliance**: Strong instruction-following and JSON formatting.

## Step 1: Install Ollama

In [ ]:
# Download and install Ollama
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]          
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates InR

## Step 2: Start Ollama background process

In [2]:
# Run Ollama serve in the background and pipe output
import subprocess
import time

print("Starting Ollama server...")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# Give it a few seconds to initialize
time.sleep(5)
print("Ollama server started!")

Starting Ollama server...


FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

## Step 3: Pull Gemma 4 Edge Model (`gemma4:e4b`)

In [ ]:
# Pull the gemma4:e4b model
!ollama pull gemma4:e4b

## Step 4: Expose the server endpoint

Choose one of the two options below to expose port `11434`.

### Option A: Localtunnel (Free, No registration required)

In [ ]:
# Install localtunnel package
!npm install -g localtunnel

# Expose Ollama port 11434. Copy the generated URL into your backend's .env as OLLAMA_API_URL
!lt --port 11434

### Option B: Ngrok (Requires free account/token)
1. Sign up at [ngrok.com](https://ngrok.com) to get your Auth Token.
2. Replace `YOUR_NGROK_AUTH_TOKEN` in the cell below and run it.

## Step 5: Update your Backend configuration
1. Copy the public URL generated from Option A or Option B (e.g., `https://xxxx.localtunnel.me` or `https://xxxx.ngrok-free.app`).
2. In your backend `.env` file, update the following configurations:
   ```env
   OLLAMA_API_URL=<YOUR_EXPOSED_URL>
   OLLAMA_MODEL=gemma4:e4b
   ```
3. Restart your Kafka consumer workers. All write actions requiring consensus checks will now be verified by this Colab GPU instance.